<a href="https://colab.research.google.com/github/imtisalrangrez/DeepLearning_Lab/blob/main/DL_week10(180).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Guided Backpropagation

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# 1. Load CIFAR-10 and Pick a Target Image
# ---------------------------------------------------------
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

x_train = x_train / 255.0
x_test  = x_test  / 255.0

# y_train/y_test are shape (N,1) in CIFAR-10 — flatten for indexing
y_train_flat = y_train.flatten()
y_test_flat  = y_test.flatten()

class_names = ['Airplane','Automobile','Bird','Cat','Deer',
               'Dog','Frog','Horse','Ship','Truck']

# Horse (label 7) — strong body outline, legs, and neck make GBP maps striking
image_idx    = np.where(y_test_flat == 7)[0][0]
target_image = tf.convert_to_tensor(x_test[image_idx:image_idx+1], dtype=tf.float32)
target_class = int(y_test_flat[image_idx])

print(f"Selected image: {class_names[target_class]} (index {image_idx})")

# ---------------------------------------------------------
# 2. Build CNN (logits output — no softmax for accurate gradients)
# ---------------------------------------------------------
def build_cnn(activation_fn='relu'):
    inputs = tf.keras.Input(shape=(32, 32, 3))             # RGB input
    x = tf.keras.layers.Conv2D(32, (3, 3), padding='same', activation=activation_fn)(inputs)
    x = tf.keras.layers.MaxPooling2D((2, 2))(x)
    x = tf.keras.layers.Conv2D(64, (3, 3), padding='same', activation=activation_fn)(x)
    x = tf.keras.layers.MaxPooling2D((2, 2))(x)
    x = tf.keras.layers.Flatten()(x)
    # Linear output — raw logits give cleaner, uncompressed gradients
    outputs = tf.keras.layers.Dense(10, activation='linear')(x)
    return tf.keras.Model(inputs, outputs)

standard_model = build_cnn(activation_fn='relu')
standard_model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

print("Training standard model to learn features...")
standard_model.fit(x_train, y_train_flat, epochs=5, batch_size=256, verbose=0)
# 5 epochs instead of 3 — CIFAR-10 needs more passes to learn meaningful color features

# ---------------------------------------------------------
# 3. Guided ReLU — blocks negative gradients during backprop
# ---------------------------------------------------------
@tf.custom_gradient
def guided_relu(x):
    """
    Forward pass: standard ReLU.
    Backward pass: only allows gradients that are both positive
                   AND flowing through positively activated neurons.
    """
    def grad(dy):
        return (tf.cast(dy > 0, dtype=tf.float32) *
                tf.cast(x  > 0, dtype=tf.float32) * dy)
    return tf.nn.relu(x), grad

# ---------------------------------------------------------
# 4. Clone Model and Inject Guided ReLU
# ---------------------------------------------------------
guided_model = build_cnn(activation_fn=guided_relu)
guided_model.set_weights(standard_model.get_weights())   # copy trained weights

# ---------------------------------------------------------
# 5. Execute Guided Backpropagation
# ---------------------------------------------------------
with tf.GradientTape() as tape:
    tape.watch(target_image)
    predictions = guided_model(target_image)
    loss = predictions[0, target_class]                  # score for the target class only

gbp_gradients = tape.gradient(loss, target_image)        # shape: (1, 32, 32, 3)

# ---------------------------------------------------------
# 6. Process Gradients for Visualization
# ---------------------------------------------------------
gbp_array = gbp_gradients[0].numpy()                     # (32, 32, 3)

# Average across RGB channels → single saliency map (32, 32)
# This collapses "which pixels mattered" across all color channels into one map
gbp_gray = np.mean(gbp_array, axis=-1)                   # (32, 32)

# Normalize to [0, 1] for clean display
gbp_gray -= gbp_gray.min()
gbp_gray /= (gbp_gray.max() + tf.keras.backend.epsilon())

# Per-channel normalization for the RGB gradient visualization
gbp_rgb = gbp_array.copy()
gbp_rgb -= gbp_rgb.min()
gbp_rgb /= (gbp_rgb.max() + tf.keras.backend.epsilon())

# ---------------------------------------------------------
# 7. Visualize — 4 panels for full insight
# ---------------------------------------------------------
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle(f'Guided Backpropagation — CIFAR-10: {class_names[target_class]}', fontsize=14)

# Panel 1: Original image
axes[0].imshow(target_image[0].numpy())
axes[0].set_title("Original Image")
axes[0].axis('off')

# Panel 2: GBP averaged saliency heatmap
im = axes[1].imshow(gbp_gray, cmap='magma')
axes[1].set_title("GBP Saliency Map\n(avg across RGB channels)")
axes[1].axis('off')
plt.colorbar(im, ax=axes[1], fraction=0.046)

# Panel 3: GBP per-channel RGB gradients
# Shows WHICH color channel each pixel's influence came from
axes[2].imshow(gbp_rgb)
axes[2].set_title("GBP RGB Gradients\n(per-channel influence)")
axes[2].axis('off')

# Panel 4: Saliency map overlaid on original
# Blend original + heatmap to see exactly which part of the horse matters
overlay = 0.6 * target_image[0].numpy() + 0.4 * plt.cm.magma(gbp_gray)[:, :, :3]
overlay = np.clip(overlay, 0, 1)
axes[3].imshow(overlay)
axes[3].set_title("Overlay\n(saliency on original)")
axes[3].axis('off')

plt.tight_layout()
plt.show()